# Clean RGI

Author: Ann Windnagel

Date: 3/10/19

Description:  
This notebook cleans up two things in the RGI data: 

1) For newer versions (those after V6) sets the shapefile attributes collumn names to be the same as V6 so the rest of the notebooks can run. 

2) This notebook cleans up Region 5 (Greenland Periphery) by removing glaciers from the database that have connectivity level of 2. We are only interested in glaciers with connectivity to the Greenland Ice Sheet of level 0 or 1. It saves the cleaned RGI database to a new file. April 2024 Note: Don't need to do this for Version 7 because it only contains glaciers with connectivity level 0 and 1. However, later notebooks do expect to read a new rgi file with "cleaned" in the file name so should run this code anyway to create that file and because of the cleaning done in #1 above.

Rastner et al. (2012) developed this connectivity level model to help describe how attached or detached a glacier is to an ice sheet and to distinguish between local glaciers around the periphery of an ice sheet from the outlet glaciers of the ice sheet. Rastner et al. (2012) created three connectivity levels for this purpose; they are described in the list below. Rastner et al. (2012) recommends that glaciers with a connectivity level of 2 be treated as part of the ice sheet. Therefore, for this study, only glaciers with connectivity level of 0 or 1 were considered.

* 0 - Indicates that the glacier is physically detached from the ice sheet and is not connected.
* 1 - Indicates that the glacier is weakly connected to the ice sheet. This means that the glacier is only in contact with the ice sheet at a well-defined divide in the accumulation zone.
* 2 - Indicates that the glacier is strongly connected. This means that the divide between the glacier and the ice sheet is indistinct in the accumulation zone and/or confluent with an ice-sheet outlet in the ablation zone.

Rastner, P., Bolch, T., Mölg, N., Machguth, H., Bris, R. L., & Paul, F. (2012). The first complete inventory of the local glaciers and ice caps on Greenland. The Cryosphere, 6(6), 1483-1495.

In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import earthpy as et

# set working dir
os.chdir(os.path.join(et.io.HOME, "git/wgms-glacier-project"))

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import scripts.wgms_scripts as ws

In [2]:
# Set version
version = 7

## Reset column names for version > 6

In [3]:
# set region numbers
region_no = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]

# Open each rgi file and clean the attribute names
for region in region_no:
    region_clean_fp = "data/rgi/processed/RGI-V" + str(version) + "/cleaned/rgi_region_" + \
                      str(region) + "_cleaned.shp"
    if os.path.exists(region_clean_fp) == False:
        print("Cleaning region: " + str(region))
        rgi_polygons = ws.open_raw_rgi(region, version)
        ws.clean_rgi(rgi_polygons, region_clean_fp, region, version)
    else:
        print(region_clean_fp + " already exists")

data/rgi/processed/RGI-V7/cleaned/rgi_region_1_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_2_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_3_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_4_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_5_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_6_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_7_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_8_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_9_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_10_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_11_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_12_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/rgi_region_13_cleaned.shp already exists
data/rgi/processed/RGI-V7/cleaned/

In [4]:
# check a cleaned RGI file
rgi_clean_polygons = ws.open_clean_rgi(5, version)
rgi_clean_polygons.head()

,RGIId,o1region,GLIMSId,BgnDate,cenlon,cenlat,Area,primeclass,Connect,Name,geometry
0,RGI2000-v7.0-G-05-00001,05,G287319E78230N,1999-07-24T00:00:00,-72.684017,78.229618,0.087347,0,0,None,"POLYGON Z ((-72.686238 78.23039799999999 0, -7..."
1,RGI2000-v7.0-G-05-00002,05,G287427E78214N,1999-07-24T00:00:00,-72.571298,78.214611,0.090965,0,0,None,"POLYGON Z ((-72.57120500000001 78.216505 0, -7..."
2,RGI2000-v7.0-G-05-00003,05,G287430E78209N,1999-07-24T00:00:00,-72.568447,78.209480,0.160309,0,0,None,"POLYGON Z ((-72.56228400000001 78.213116 0, -7..."
3,RGI2000-v7.0-G-05-00004,05,G287500E78199N,1999-07-24T00:00:00,-72.554216,78.197976,0.470115,0,0,None,"POLYGON Z ((-72.532769 78.20244099999999 0, -7..."
4,RGI2000-v7.0-G-05-00005,05,G287465E78186N,1999-07-24T00:00:00,-72.532406,78.185615,0.133281,0,0,None,"POLYGON Z ((-72.533734 78.187888 0, -72.533576..."


## Extra Cleaning for Region 5: Greenland Periphery

In [5]:
# Open cleaned RGI region 05 - Greenland Periphery
rgi_region05_polygons = ws.open_clean_rgi(5, version)

In [6]:
# Check number of polygons in the pre-cleaned dataframe
preclean = len(rgi_region05_polygons)
preclean

19994

In [7]:
# Select glaciers that have connectivity level of 0 or 1
rgi_region05_polygons = rgi_region05_polygons.loc[
    (rgi_region05_polygons['Connect'] == 0) | (rgi_region05_polygons['Connect'] == 1)]

In [8]:
# Check the number of resulting polygons in the clean dataframe
postclean = len(rgi_region05_polygons)
postclean

19994

In [9]:
# Save new shapefile if needed
if preclean == postclean:
    # Cleaning already done or not needed
    print("The pre-clean and post-clean polygon counts are the same. No cleaning needed.")
elif preclean > postclean:
    # Write dataframe to shapefile
    clean_fn = "data/rgi/processed/RGI-V" + str(version) + "/cleaned/rgi_region_5_cleaned.shp"
    rgi_region05_polygons.to_file(driver='ESRI Shapefile', filename=clean_fn)
    print("Writing cleaned region 5 shapefile")

The pre-clean and post-clean polygon counts are the same. No cleaning needed.


In [13]:
# Open the new shapefile to make sure it is okay.
rgi_region05_clean_polygons = ws.open_clean_rgi(5, version)
rgi_region05_clean_polygons.head()

,RGIId,o1region,GLIMSId,BgnDate,cenlon,cenlat,Area,primeclass,Connect,Name,geometry
0,RGI2000-v7.0-G-05-00001,05,G287319E78230N,1999-07-24T00:00:00,-72.684017,78.229618,0.087347,0,0,None,"POLYGON Z ((-72.686238 78.23039799999999 0, -7..."
1,RGI2000-v7.0-G-05-00002,05,G287427E78214N,1999-07-24T00:00:00,-72.571298,78.214611,0.090965,0,0,None,"POLYGON Z ((-72.57120500000001 78.216505 0, -7..."
2,RGI2000-v7.0-G-05-00003,05,G287430E78209N,1999-07-24T00:00:00,-72.568447,78.209480,0.160309,0,0,None,"POLYGON Z ((-72.56228400000001 78.213116 0, -7..."
3,RGI2000-v7.0-G-05-00004,05,G287500E78199N,1999-07-24T00:00:00,-72.554216,78.197976,0.470115,0,0,None,"POLYGON Z ((-72.532769 78.20244099999999 0, -7..."
4,RGI2000-v7.0-G-05-00005,05,G287465E78186N,1999-07-24T00:00:00,-72.532406,78.185615,0.133281,0,0,None,"POLYGON Z ((-72.533734 78.187888 0, -72.533576..."


In [11]:
# Read the region directories and select region 5 (index 4)
# Old unneeded code below
# This code reads a list of files from a directory and then creates a "clean" file name from that
#root_data_dir = "data/rgi/raw/RGI-V" + str(version)
#subdirs = [ f.path for f in os.scandir(root_data_dir) if f.is_dir() ]
#subsplit = subdirs[4].rsplit('\\')
#path = subdirs[4].replace('\\', '/') + '/' + subsplit[-1] + '.shp'

# Insert the text '_clean' into the filename
#addtext = '_clean'
#text1 = '.shp'
#text1ans = path.find(text1)
#clean_fn_draft = path[:text1ans] + newText + path[text1ans:]

#text2 = 'periphery/'
#text2ans = clean_fn_draft.find(text2)
#clean_fn = clean_fn_draft[:text2ans + len(text2)-1] + newText + clean_fn_draft[text2ans + len(text2)-1:]
#clean_fn